# Notebook 07 — Sim-to-sim: calibrating MuJoCo against Gazebo, and predicting failures

Companion to [Lecture 8](../08_mujoco_simulation.md) (§5–§6). We run the same identification drive in MuJoCo
that was run in Gazebo (`../data/sysid_gazebo.csv`), calibrate tyre friction, reproduce brake-induced
oversteer, and compare policies across the bicycle model, MuJoCo and Gazebo.

In [ ]:
import os; os.environ.setdefault('MUJOCO_GL', 'egl')          # offscreen rendering (GPU); use 'osmesa' without one
import warnings; warnings.filterwarnings('ignore', message='Unable to import Axes3D')
import sys, math
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, mujoco
WS = Path('../../..').resolve(); sys.path.insert(0, str(WS / 'src' / 'saurcon_rl')); sys.path.insert(0, str(WS / 'src' / 'saurcon_plan'))
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.grid': True, 'grid.color': '#e4e3df',
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE, ORANGE, AQUA, GREY = '#2a78d6', '#eb6834', '#1baf7a', '#52514e'
print('MuJoCo', mujoco.__version__)
from saurcon_rl.sysid import mujoco_main
from saurcon_rl.sysid_compare import analyse
import saurcon_rl.mujoco_model as MM

## 1. The identification drive in MuJoCo, for several friction values (§5)

In [ ]:
import csv, tempfile
gz = analyse('../data/sysid_gazebo.csv')
results = {}
original = MM.build_mjcf
for mu in (1.2, 1.4, 1.6, 1.8):
    MM.build_mjcf = lambda *a, mu=mu, **k: original(*a, **{**k, 'friction': mu})
    import saurcon_rl.sysid as S; S_build = None
    rows = mujoco_main(Path(tempfile.mkdtemp()))
    path = Path(tempfile.mkdtemp()) / 'sysid.csv'
    with open(path, 'w', newline='') as f: w = csv.writer(f); w.writerow(['t','cmd_v','cmd_steer','x','y','yaw']); w.writerows(rows)
    results[mu] = analyse(path)
MM.build_mjcf = original
keys = [k for k in gz[2] if k[0] <= 3]          # steady rows: at 4 m/s and above both cars are spinning
x = np.arange(len(keys))
plt.figure(figsize=(10, 4))
plt.plot(x, [gz[2][k][2] for k in keys], 'k-o', label='Gazebo')
for (mu, res), c in zip(results.items(), (ORANGE, BLUE, AQUA, GREY)):
    plt.plot(x, [res[2][k][2] for k in keys], '-o', color=c, ms=4, label=f'MuJoCo mu={mu}')
plt.xticks(x, [f'{v}/{s}' for v, s in keys], rotation=60); plt.xlabel('commanded speed (m/s) / steering (rad)')
plt.ylabel('steady lateral accel (m/s^2)'); plt.legend(); plt.title('Calibrating MuJoCo tyre friction against Gazebo'); plt.show()
for mu, res in results.items():
    diff = [res[2][k][2] - gz[2][k][2] for k in keys]
    print(f'mu={mu}: accel {res[0]:.2f} (Gazebo {gz[0]:.2f}), brake {res[1]:.2f} (Gazebo {gz[1]:.2f}), '
          f'lateral RMS diff {np.sqrt(np.mean(np.square(diff))):.2f}, mean {np.mean(diff):+.2f} m/s^2')

Only the steady 2–3 m/s rows are compared: at 4 m/s and above both cars spin, and the numbers are transient. The
metric decides the answer (Lecture 8 §5). $\mu = 1.4$ fits best and sits slightly *under* Gazebo's grip on
average, the safe side for training: a policy that expects a little less grip than it gets will not overdrive.

## 2. Brake-induced oversteer (§6)
At 6.5 m/s with a small 0.04 rad steering input, compare coasting (speed held) with braking at 4 m/s². The bicycle
model predicts the same yaw rate, $v\tan\delta/L$, either way.

In [ ]:
from saurcon_rl.mujoco_model import build_mjcf, ackermann, WHEEL_R, TRACK, WHEELBASE
def manoeuvre(brake):
    mm = mujoco.MjModel.from_xml_string(build_mjcf()); dd = mujoco.MjData(mm); h = mm.opt.timestep
    v = 0.0; out = []
    for k in range(int(5.0 / h)):
        t = k * h
        target = 6.5 if t < 3.0 or not brake else max(0.0, 6.5 - 4.0*(t - 3.0))
        v += np.clip(target - v, -4.0*h, 3.8*h)
        delta = 0.04 if t >= 3.0 else 0.0
        l, r = ackermann(delta); kap = math.tan(delta)/WHEELBASE; w = v / WHEEL_R
        dd.ctrl[:] = [w*(1 - kap*TRACK/2), w*(1 + kap*TRACK/2), l, r]; mujoco.mj_step(mm, dd)
        q = dd.qpos[3:7]; yaw = math.atan2(2*(q[0]*q[3]), 1 - 2*q[3]**2)
        vx, vy = dd.qvel[0], dd.qvel[1]; vl = math.cos(yaw)*vx + math.sin(yaw)*vy; vt = -math.sin(yaw)*vx + math.cos(yaw)*vy
        out.append((t, vl, dd.qvel[5], math.degrees(math.atan2(vt, max(abs(vl), 1e-3)))))
    return np.array(out)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for brake, c, lab in ((False, BLUE, 'coast at 6.5 m/s'), (True, ORANGE, 'brake at 4 m/s^2')):
    L = manoeuvre(brake); s = L[:, 0] >= 2.5
    ax[0].plot(L[s, 0], L[s, 2], color=c, label=lab); ax[1].plot(L[s, 0], L[s, 3], color=c, label=lab)
    ax[0].plot(L[s, 0], L[s, 1]*math.tan(0.04)/WHEELBASE*(L[s, 0] >= 3), color=c, ls=':', lw=1)
ax[0].set_ylabel('yaw rate (rad/s)  (dotted: bicycle model)'); ax[1].set_ylabel('sideslip (deg)')
for a in ax: a.set_xlabel('time (s)'); a.legend()
fig.suptitle('Same steering input; braking with rear-only brakes makes the car rotate more and slide'); plt.show()

## 3. One policy, three simulators (§6)

In [ ]:
from saurcon_rl.env import RaceEnv, load_tracks
from saurcon_rl.mujoco_env import MujocoRaceEnv
from saurcon_rl.train import evaluate
from stable_baselines3 import PPO
gazebo = {'ppo_v1': 'spun in the far bend (full loop, 2026-11-12)',
          'ppo_v2': 'spun at the end of the first bend (full loop, 2026-11-12)'}
rows = []
for name in ('ppo_v1', 'ppo_v2', 'ppo_mj1'):
    p = WS / 'rl_runs' / name / 'best_model.zip'
    if not p.exists(): continue
    m = PPO.load(p, device='cpu'); res = []
    for cls in (RaceEnv, MujocoRaceEnv):
        r = evaluate(m, cls(load_tracks(), randomize=False, max_time=200))
        res.append(f"{r['flying_lap_s']} s lap" if r['flying_lap_s'] and not r['crashed'] else f"crash at {r['progress_m']} m")
    rows.append((name, *res, gazebo.get(name, 'see src/saurcon_rl/README.md')))
print(f"{'policy':10s} {'bicycle model':18s} {'MuJoCo':18s} Gazebo")
for r in rows: print(f'{r[0]:10s} {r[1]:18s} {r[2]:18s} {r[3]}')

## 4. A MuJoCo-trained policy driving

In [ ]:
p = WS / 'rl_runs' / 'ppo_mj1' / 'best_model.zip'
if p.exists():
    from saurcon_rl.train import race_start_s
    m = PPO.load(p, device='cpu'); env = MujocoRaceEnv(load_tracks(), randomize=False, max_time=60, cone_mode='visual')
    obs, _ = env.reset(options={'track': 0, 'start_s': race_start_s(env.tracks[0])})
    rend = mujoco.Renderer(env.m, 270, 480); frames = []; log = []
    for k in range(int(12 / 0.05)):
        a, _ = m.predict(obs, deterministic=True); obs, _, term, trunc, info = env.step(a)
        log.append((env.t, env.v, info['slip_deg']))
        if k % 30 == 0: rend.update_scene(env.d, camera='chase'); frames.append(rend.render())
        if term: break
    fig, ax = plt.subplots(2, 4, figsize=(14, 5))
    for a, f in zip(ax.ravel(), frames): a.imshow(f); a.axis('off')
    plt.suptitle('MuJoCo-trained policy, chase camera every 1.5 s'); plt.show()
    L = np.array(log); plt.plot(L[:, 0], L[:, 1], color=BLUE, label='speed (m/s)'); plt.plot(L[:, 0], L[:, 2], color=ORANGE, label='sideslip (deg)')
    plt.xlabel('time (s)'); plt.legend(); plt.show(); print('lap times so far:', info['lap_times'])
else:
    print('Train first: python -m saurcon_rl.train --env mujoco --out rl_runs/ppo_mj1 ...')

## Problems
Lecture 8's problems **8.4** and **8.6** build on this notebook.

**N7.1** Repeat section 2 with braking at 2 m/s². Where is the threshold for losing the rear?

**N7.2** In section 3, evaluate the policies with `randomize=True` over 20 seeds and report crash rates with
Clopper–Pearson intervals (Lecture 7 §5).

**N7.3** Add front brakes to the model (problem 8.4) and retrain briefly. How much lap time does it gain, and does
the bicycle model's verdict now agree with MuJoCo's?